# Учебный мини-проект: антифрод логинов

## Цель

Из таблицы сырых событий построить leakage-safe признаки, разбить данные по времени и обучить модель, которая оценивает вероятность мошеннического логина.

**Единица классификации:** один логин.  
**Момент прогнозирования:** сразу после завершения попытки аутентификации.  
**Таргет:** `is_fraud`. Он подтверждается позже и не может использоваться как признак.

Начни с этого ноутбука. Когда застрянешь, открой `antifraud_solution.ipynb`.

## 1. Setup

Ноутбук ожидает, что `raw_login_events.csv` лежит рядом с ним.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    PrecisionRecallDisplay,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("raw_login_events.csv")
RANDOM_STATE = 42

## 2. Загрузка и первичная проверка

В сырых данных уже есть таргет, чтобы упражнение можно было выполнить автономно.  
Колонка `fraud_confirmed_at` показывает задержку разметки. Ее нельзя использовать как признак.

In [ ]:
raw = pd.read_csv(
    DATA_PATH,
    parse_dates=["event_time", "fraud_confirmed_at"],
)

print("Строк:", len(raw))
print("Доля fraud:", raw["is_fraud"].mean().round(4))
raw.head()

In [ ]:
# Базовые проверки качества данных.
assert raw["event_id"].is_unique
assert raw["event_time"].notna().all()
assert set(raw["is_fraud"].unique()) <= {0, 1}

raw.info()

## 3. Построй функции для признаков из прошлого

Главное правило:

```text
feature(event at time t) использует только события со временем < t
```

Не используй `groupby(...).transform(...)` по всей таблице без временного окна: так в строку попадет будущее.

In [ ]:

from collections import defaultdict, deque, Counter

def past_count(
    frame,
    group_col,
    window,
    condition_col=None,
    condition_value=None,
):
    """
    TODO:
    Return the number of PREVIOUS events in
    [event_time - window, event_time).

    Hint:
    1. Iterate over rows sorted by event_time.
    2. Keep one deque for each group value.
    3. Remove expired events.
    4. Calculate the feature.
    5. Only then append the current event.
    """
    raise NotImplementedError("Complete past_count")


def past_unique(frame, group_col, value_col, window):
    """
    TODO:
    Count distinct past value_col values for every group_col value
    inside the selected time window.

    Hint:
    Use a deque plus collections.Counter.
    """
    raise NotImplementedError("Complete past_unique")


def seen_before(frame, group_col, value_col):
    """
    TODO:
    Return 1 when a value has appeared for this group before,
    otherwise return 0.

    Important: add the current value to history only after computing
    the current row.
    """
    raise NotImplementedError("Complete seen_before")


def seconds_since_previous(frame, group_col, cap_seconds=30 * 86400):
    """TODO: seconds since the previous event for the group."""
    raise NotImplementedError("Complete seconds_since_previous")


## 4. Собери модельную таблицу

Рекомендуемые признаки:

- число прошлых событий пользователя за 1 час и 24 часа;
- число прошлых неуспешных попыток за 24 часа;
- число уникальных IP пользователя за 7 дней;
- активность и число пользователей на IP за 1 час;
- число пользователей устройства за 24 часа;
- встречались ли устройство, IP и страна у пользователя раньше;
- время с предыдущего события пользователя.

In [ ]:

def build_features(raw_frame):
    frame = (
        raw_frame
        .sort_values(["event_time", "event_id"])
        .reset_index(drop=True)
        .copy()
    )

    # TODO 1: current-event time features.
    frame["hour"] = ...
    frame["is_night"] = ...

    # TODO 2: user history features.
    frame["user_events_1h"] = ...
    frame["user_events_24h"] = ...
    frame["user_failed_24h"] = ...
    frame["user_unique_ips_7d"] = ...

    # TODO 3: IP/device features.
    frame["ip_events_1h"] = ...
    frame["ip_unique_users_1h"] = ...
    frame["device_unique_users_24h"] = ...

    # TODO 4: familiarity features.
    frame["seen_device_before"] = ...
    frame["seen_ip_before"] = ...
    frame["seen_country_before"] = ...
    frame["seconds_since_prev_user_event"] = ...

    return frame


In [ ]:
model_frame = build_features(raw)
model_frame.head()

## 5. Leakage-check на одной строке

Выбери событие и вручную проверь один признак. Например, `user_events_24h` должен совпасть с числом событий того же пользователя в предыдущие 24 часа.

In [ ]:
row_number = 5000
event = model_frame.iloc[row_number]

manual_history = model_frame[
    (model_frame["user_id"] == event["user_id"])
    & (model_frame["event_time"] < event["event_time"])
    & (
        model_frame["event_time"]
        >= event["event_time"] - pd.Timedelta(days=1)
    )
]

print("Значение признака:", event["user_events_24h"])
print("Ручной расчет:", len(manual_history))
assert event["user_events_24h"] == len(manual_history)

## 6. Временной split

Не перемешивай строки. Используй прошлое для обучения, более поздний период для валидации и самый поздний — для теста.

In [ ]:
TRAIN_END = pd.Timestamp("2025-02-20")
VALID_END = pd.Timestamp("2025-03-10")

# TODO: создай train, valid и test по event_time.
train = ...
valid = ...
test = ...

for name, part in [("train", train), ("valid", valid), ("test", test)]:
    print(name, len(part), part["is_fraud"].mean().round(4))

## 7. Обучи baseline

Не передавай в модель:

- `is_fraud`;
- `fraud_confirmed_at`;
- `event_id`;
- временную метку как необработанную строку;
- сырые `user_id`, `ip_id`, `device_id` в первом baseline.

Начни с логистической регрессии с `class_weight="balanced"`.

In [ ]:
numeric_features = [
    "hour",
    "is_night",
    "login_success",
    "user_events_1h",
    "user_events_24h",
    "user_failed_24h",
    "user_unique_ips_7d",
    "ip_events_1h",
    "ip_unique_users_1h",
    "device_unique_users_24h",
    "seen_device_before",
    "seen_ip_before",
    "seen_country_before",
    "seconds_since_prev_user_event",
]
categorical_features = ["auth_method", "country"]
feature_columns = numeric_features + categorical_features

# TODO:
# 1. ColumnTransformer для numeric и categorical.
# 2. Pipeline.
# 3. model.fit(train[feature_columns], train["is_fraud"])
model = ...

## 8. Оцени ранжирование и выбери порог

Для редкого класса важнее `PR-AUC`, чем accuracy.  
Затем выбери порог на validation. Ниже предлагается бизнес-ограничение: отправлять на проверку около 5% событий.

In [ ]:
valid_score = model.predict_proba(valid[feature_columns])[:, 1]

print("Validation PR-AUC:", average_precision_score(
    valid["is_fraud"], valid_score
))
print("Validation ROC-AUC:", roc_auc_score(
    valid["is_fraud"], valid_score
))

threshold = np.quantile(valid_score, 0.95)
print("Threshold:", threshold)

In [ ]:
def print_metrics(y_true, score, threshold):
    prediction = (score >= threshold).astype(int)
    print("PR-AUC:", round(average_precision_score(y_true, score), 4))
    print("ROC-AUC:", round(roc_auc_score(y_true, score), 4))
    print("Precision:", round(precision_score(y_true, prediction), 4))
    print("Recall:", round(recall_score(y_true, prediction), 4))
    print("F1:", round(f1_score(y_true, prediction), 4))
    print("Alert rate:", round(prediction.mean(), 4))
    print("Confusion matrix:\n", confusion_matrix(y_true, prediction))

test_score = model.predict_proba(test[feature_columns])[:, 1]
print_metrics(test["is_fraud"], test_score, threshold)

## 9. Что попробовать дальше

1. Сравнить пороги для alert budget 1%, 3%, 5% и 10%.
2. Добавить окна 10 минут, 6 часов и 30 дней.
3. Добавить число стран пользователя за 30 дней.
4. Обучить градиентный бустинг и сравнить его честно на том же test.
5. Провести rolling validation по нескольким временным блокам.